In [1]:
import pandas as pd
import numpy as np

In [2]:
customers = pd.read_csv("customers.csv")

customers.head()

,CustomerID,Age,Gender,Occupation,MonthlyIncome,Region,Branch,FCYAccount,ETBAccount,MobileBanking,InternetBanking,RemittanceCount,TotalRemittanceUSD,AverageTransactionUSD,FCYPurchaseUSD,LastTransactionDays,ExistingProducts,LeadStatus
0,100001,59,Female,Business Owner,71321,Addis Ababa,Adama,Yes,Yes,Yes,No,5,5145,857.50,2167,166,2,0
1,100002,49,Male,Merchant,144028,Amhara,CMC,No,Yes,No,No,8,8416,935.11,1779,150,5,0
2,100003,35,Female,Teacher,86131,Addis Ababa,Bole,No,Yes,Yes,No,5,5925,987.50,2163,363,2,0
3,100004,63,Female,Merchant,37963,Tigray,Megenagna,Yes,Yes,No,Yes,5,3815,635.83,1569,343,3,0
4,100005,28,Male,Private Employee,113757,Amhara,CMC,No,Yes,No,No,3,3033,758.25,1615,68,3,0


In [3]:
customers.columns

Index(['CustomerID', 'Age', 'Gender', 'Occupation', 'MonthlyIncome', 'Region',
       'Branch', 'FCYAccount', 'ETBAccount', 'MobileBanking',
       'InternetBanking', 'RemittanceCount', 'TotalRemittanceUSD',
       'AverageTransactionUSD', 'FCYPurchaseUSD', 'LastTransactionDays',
       'ExistingProducts', 'LeadStatus'],
      dtype='object')

In [4]:
#Customer Tenure Category
customers["AgeGroup"] = pd.cut(
    customers["Age"],
    bins=[20,30,40,50,60,70],
    labels=[
        "21-30",
        "31-40",
        "41-50",
        "51-60",
        "61-70"
    ]
)

In [5]:
#High Income Customer
customers["HighIncome"] = np.where(
    customers["MonthlyIncome"] >= 80000,
    1,
    0
)

In [6]:
#Frequent Remittance Customer
customers["FrequentRemittance"] = np.where(
    customers["RemittanceCount"] >= 8,
    1,
    0
)

In [7]:
#High Value Remittance

customers["HighRemittance"] = np.where(
    customers["TotalRemittanceUSD"] >= 5000,
    1,
    0
)


In [8]:
#Active Customer

customers["ActiveCustomer"] = np.where(
    customers["LastTransactionDays"] <= 30,
    1,
    0
)

In [9]:
#Digital Customer

customers["DigitalCustomer"] = np.where(
    (
        (customers["MobileBanking"]=="Yes") &
        (customers["InternetBanking"]=="Yes")
    ),
    1,
    0
)

In [10]:
#Customer Value Score

customers["CustomerValueScore"] = (
      customers["MonthlyIncome"]*0.30
    + customers["TotalRemittanceUSD"]*0.50
    + customers["ExistingProducts"]*1000*0.20
)


In [11]:
#Average Remittance Per Month

customers["MonthlyRemittanceAverage"] = (
    customers["TotalRemittanceUSD"] / 12
).round(2)

In [12]:
#FCY Utilization Rate

customers["FCYUtilizationRate"] = (
    customers["FCYPurchaseUSD"] /
    customers["TotalRemittanceUSD"]
).round(2)

customers["FCYUtilizationRate"] = np.where(
    customers["TotalRemittanceUSD"] == 0,
    0,
    customers["FCYPurchaseUSD"] /
    customers["TotalRemittanceUSD"]
)

In [13]:
#Banking Relationship Score

customers["RelationshipScore"] = (
    customers["ETBAccount"].map({"Yes":1,"No":0}) +
    customers["MobileBanking"].map({"Yes":1,"No":0}) +
    customers["InternetBanking"].map({"Yes":1,"No":0}) +
    customers["ExistingProducts"]
)

In [14]:
#Potential FCY Customer

customers["PotentialFCYCustomer"] = np.where(
    (
        (customers["FCYAccount"]=="No") &
        (customers["HighIncome"]==1) &
        (customers["FrequentRemittance"]==1)
    ),
    1,
    0
)


In [15]:
#Review the New Features

customers.head()

,CustomerID,Age,Gender,Occupation,MonthlyIncome,Region,Branch,FCYAccount,ETBAccount,MobileBanking,...,HighIncome,FrequentRemittance,HighRemittance,ActiveCustomer,DigitalCustomer,CustomerValueScore,MonthlyRemittanceAverage,FCYUtilizationRate,RelationshipScore,PotentialFCYCustomer
0,100001,59,Female,Business Owner,71321,Addis Ababa,Adama,Yes,Yes,Yes,...,0,0,1,0,0,24368.8,428.75,0.421186,4,0
1,100002,49,Male,Merchant,144028,Amhara,CMC,No,Yes,No,...,1,1,1,0,0,48416.4,701.33,0.211383,6,1
2,100003,35,Female,Teacher,86131,Addis Ababa,Bole,No,Yes,Yes,...,1,0,1,0,0,29201.8,493.75,0.365063,4,0
3,100004,63,Female,Merchant,37963,Tigray,Megenagna,Yes,Yes,No,...,0,0,0,0,0,13896.4,317.92,0.411271,5,0
4,100005,28,Male,Private Employee,113757,Amhara,CMC,No,Yes,No,...,1,0,0,0,0,36243.6,252.75,0.532476,4,0


In [16]:
#Verify No Missing Values

customers.isnull().sum()

CustomerID                  0
Age                         0
Gender                      0
Occupation                  0
MonthlyIncome               0
Region                      0
Branch                      0
FCYAccount                  0
ETBAccount                  0
MobileBanking               0
InternetBanking             0
RemittanceCount             0
TotalRemittanceUSD          0
AverageTransactionUSD       0
FCYPurchaseUSD              0
LastTransactionDays         0
ExistingProducts            0
LeadStatus                  0
AgeGroup                    0
HighIncome                  0
FrequentRemittance          0
HighRemittance              0
ActiveCustomer              0
DigitalCustomer             0
CustomerValueScore          0
MonthlyRemittanceAverage    0
FCYUtilizationRate          0
RelationshipScore           0
PotentialFCYCustomer        0
dtype: int64

In [17]:
#Explore the New Features

new_features = [
    "CustomerValueScore",
    "RelationshipScore",
    "FCYUtilizationRate",
    "PotentialFCYCustomer"
]

customers[new_features].describe()

,CustomerValueScore,RelationshipScore,FCYUtilizationRate,PotentialFCYCustomer
count,500.000000,500.000000,500.000000,500.000000
mean,32125.505800,5.228000,0.565645,0.176000
std,14790.787369,1.515457,0.210965,0.381202
min,4886.600000,2.000000,0.200204,0.000000
25%,18550.750000,4.000000,0.388793,0.000000
50%,31969.950000,5.000000,0.561766,0.000000
75%,45053.125000,6.000000,0.736005,0.000000
max,58751.000000,8.000000,0.946956,1.000000


In [18]:
#Save the Enhanced Dataset
customers.to_csv(
    "customers_feature_engineered.csv",
    index=False
)

print("Feature engineered dataset saved successfully!")

Feature engineered dataset saved successfully!


In [19]:
#Business Validation

customers["PotentialFCYCustomer"].value_counts()
customers["HighIncome"].value_counts()
customers["DigitalCustomer"].value_counts()
customers["CustomerValueScore"].describe()

count      500.000000
mean     32125.505800
std      14790.787369
min       4886.600000
25%      18550.750000
50%      31969.950000
75%      45053.125000
max      58751.000000
Name: CustomerValueScore, dtype: float64

Final Dataset

Your dataset now contains:

Original Features	New Engineered Features
18	11

Total = 29 Columns

Why These Features Matter
Feature	Why It Helps the Model
HighIncome	Higher-income customers often have greater FCY needs.
FrequentRemittance	Frequent remittance receivers are strong mobilization candidates.
HighRemittance	Captures customers handling large foreign currency volumes.
ActiveCustomer	Recently active customers are easier to engage.
DigitalCustomer	Measures digital engagement, which often correlates with responsiveness.
CustomerValueScore	Combines financial value into one metric.
RelationshipScore	Represents how connected the customer is to the bank.
PotentialFCYCustomer	Flags customers matching key business criteria.
FCYUtilizationRate	Shows how much of received FCY is actually used.
MonthlyRemittanceAverage	Normalizes annual remittance into a monthly measure.
AgeGroup	Captures non-linear age effects that raw age may miss.